# Phase 1: PINN Basin Maps on GPU

2D basin map (seed × learning_rate) for 1D convection PINN.

**Before running:** Runtime → Change runtime type → **GPU** (A100 or T4)

**Workflow:**
1. Run cells 1–4 (setup + verification) — should take < 2 min
2. If verification passes, run cell 5+ (production)

In [ ]:
# 1. Setup
import os
if not os.path.exists('math-ai'):
    !git clone https://github.com/dzmitrybudzko/math-ai.git
else:
    !cd math-ai && git pull

import sys
sys.path.insert(0, 'math-ai/experiments/phase1')

import torch
import numpy as np
import time

print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memory: {mem_gb:.1f} GB')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {device}')

In [ ]:
# 2. Imports + CUDA warmup
from convection_pinn import (
    train_single, compute_basin_map_2d, plot_basin_2d,
    basin_entropy, save_results, summarize_outcomes,
    CORRECT, TRIVIAL, DIVERGED
)

# Warmup CUDA
if device == 'cuda':
    _ = torch.randn(256, 256, device=device) @ torch.randn(256, 256, device=device)
    torch.cuda.synchronize()

# Time a single run
t0 = time.time()
outcome, l2, loss = train_single(seed=0, beta=1.0, method='adam', n_adam=2000, device=device)
dt1 = time.time() - t0

t0 = time.time()
outcome, l2, loss = train_single(seed=1, beta=1.0, method='adam', n_adam=2000, device=device)
dt2 = time.time() - t0

labels = {CORRECT: 'CORRECT', TRIVIAL: 'TRIVIAL', DIVERGED: 'DIVERGED'}
print(f'Run 1 (cold): {dt1:.1f}s | Run 2 (warm): {dt2:.1f}s')
print(f'Outcome: {labels[outcome]}, L2={l2:.4f}')
print(f'\nEstimated time per grid size:')
for n in [25, 100, 900, 2500]:
    print(f'  {n:5d} runs -> ~{n*dt2/60:.0f} min')

In [ ]:
# 3. VERIFICATION: tiny 5x5 grid, beta=1 — should finish in ~30s
print('=== Verification run: 5 seeds x 5 LRs, beta=1 ===')

lr_test = np.logspace(-4, -2, 5)
t0 = time.time()
out_test, l2_test, lr_test = compute_basin_map_2d(
    beta=1.0, method='adam', n_seeds=5,
    lr_values=lr_test, n_adam=2000, device=device
)
dt = time.time() - t0

print(f'\nDone in {dt:.0f}s')
summarize_outcomes(out_test)

S_test = basin_entropy(out_test, box_size=3)
print(f'Basin entropy (box=3): {S_test:.4f}')

# Verify no -1 sentinel left
assert np.all(out_test >= 0), 'BUG: unprocessed cells remain!'
print('\n✓ All cells computed, no sentinel values. Verification passed.')

In [ ]:
# 4. VERIFICATION: plot the tiny grid to check visualization
plot_basin_2d(out_test, lr_test,
              title='Verification: 5x5, beta=1',
              filename='verify_basin_5x5.png')

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

colors_map = {0: [0,0.8,0], 1: [1,0.6,0], 2: [0.2,0.2,0.2]}
h, w = out_test.shape
img = np.zeros((h, w, 3))
for code, color in colors_map.items():
    img[out_test == code] = color

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(img, aspect='auto', origin='lower',
          extent=[np.log10(lr_test[0]), np.log10(lr_test[-1]), 0, h])
ax.set_xlabel('log10(lr)')
ax.set_ylabel('Seed')
ax.set_title(f'Verification 5x5, beta=1, S_b={S_test:.3f}')
plt.show()
print('✓ If you see a colored grid above, visualization works.')

## Production runs

After verification passes, run the cells below. Each beta saves results to `.npz` + `.png` independently, so if Colab disconnects you don't lose earlier betas.

**Grid:** 30 seeds × 30 learning rates = 900 runs per beta (~107 min each at 7.2s/run).
**Betas:** 1, 2, 3, 5, 10, 30 — from easy to hard, to find the sweet spot with fractal basin boundaries.
**Total:** ~10.5 hours on A100.

In [ ]:
# 5. Production: multi-beta sweep with checkpointing
import os

BETAS = [1.0, 2.0, 3.0, 5.0, 10.0, 30.0]
N_SEEDS = 30
N_LR = 30
N_ADAM = 2000
LR_VALUES = np.logspace(-5, -1, N_LR)

OUT_DIR = 'results_phase1'
os.makedirs(OUT_DIR, exist_ok=True)

summary = {}

for beta in BETAS:
    tag = f'basin_2d_adam_beta{int(beta)}'
    npz_path = f'{OUT_DIR}/{tag}.npz'
    png_path = f'{OUT_DIR}/{tag}.png'

    # Skip if already computed
    if os.path.exists(npz_path):
        data = dict(np.load(npz_path, allow_pickle=True))
        out = data['outcomes']
        if np.all(out >= 0):
            S_b = float(data.get('basin_entropy', np.nan))
            n_c = np.sum(out == CORRECT)
            summary[beta] = {'outcomes': out, 'entropy': S_b}
            print(f'Beta={beta:.0f}: already done — {n_c}/{out.size} correct, S_b={S_b:.4f}. Skipping.')
            continue

    print(f'\n{"="*60}')
    print(f'Beta = {beta}  |  {N_SEEDS}x{N_LR} = {N_SEEDS*N_LR} runs')
    print(f'{"="*60}')

    t0 = time.time()
    out, l2e, lrs = compute_basin_map_2d(
        beta=beta, method='adam', n_seeds=N_SEEDS,
        lr_values=LR_VALUES, n_adam=N_ADAM, device=device
    )
    dt = time.time() - t0

    S_b = basin_entropy(out)
    summarize_outcomes(out)
    print(f'Basin entropy S_b = {S_b:.4f}')
    print(f'Time: {dt:.0f}s ({dt/60:.1f} min)')

    save_results(npz_path, out, l2e, lrs, beta,
                 method='adam', n_adam=N_ADAM, S_b=S_b)

    plot_basin_2d(out, lrs,
                  title=f'2D Basin: Adam, beta={int(beta)}, {N_ADAM} steps (S_b={S_b:.3f})',
                  filename=png_path)

    summary[beta] = {'outcomes': out, 'entropy': S_b}

# Final summary
print(f'\n\n{"="*60}')
print('SUMMARY')
print(f'{"="*60}')
print(f'{"beta":>6} | {"correct%":>8} | {"trivial%":>8} | {"diverged%":>9} | {"S_b":>6}')
print('-' * 52)
for b in sorted(summary.keys()):
    r = summary[b]
    o = r['outcomes']
    done = o[o >= 0]
    total = len(done)
    nc = np.sum(done == CORRECT)
    nt = np.sum(done == TRIVIAL)
    nd = np.sum(done == DIVERGED)
    print(f'{b:6.0f} | {100*nc/total:7.1f}% | {100*nt/total:7.1f}% | {100*nd/total:8.1f}% | {r["entropy"]:.4f}')

In [ ]:
# 6. Composite figure: all betas side by side
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

colors = {0: [0, 0.8, 0], 1: [1, 0.6, 0], 2: [0.2, 0.2, 0.2]}

for idx, beta in enumerate(sorted(summary.keys())):
    ax = axes[idx]
    o = summary[beta]['outcomes']
    S_b = summary[beta]['entropy']
    h, w = o.shape
    img = np.zeros((h, w, 3))
    for code, color in colors.items():
        img[o == code] = color

    ax.imshow(img, aspect='auto', origin='lower',
              extent=[np.log10(LR_VALUES[0]), np.log10(LR_VALUES[-1]), 0, h])
    nc = np.sum(o[o >= 0] == CORRECT)
    total = np.sum(o >= 0)
    ax.set_title(f'β={int(beta)}, correct={100*nc/total:.0f}%, S_b={S_b:.3f}')
    ax.set_xlabel('log₁₀(lr)')
    ax.set_ylabel('Seed')

legend = [Patch(facecolor=colors[0], label='Correct'),
          Patch(facecolor=colors[1], label='Trivial'),
          Patch(facecolor=colors[2], label='Diverged')]
fig.legend(handles=legend, loc='lower center', ncol=3, fontsize=12)
fig.suptitle('Basin Maps: Adam, 2000 steps, 30x30 grid', fontsize=14)
fig.tight_layout(rect=[0, 0.05, 1, 0.95])
fig.savefig(f'{OUT_DIR}/composite_all_betas.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUT_DIR}/composite_all_betas.png')

In [ ]:
# 7. Download all results
from google.colab import files
import glob as globmod

for f in sorted(globmod.glob(f'{OUT_DIR}/*')):
    print(f'Downloading: {f}')
    files.download(f)